In [1]:
from data_labeling_pipeline import DataLabelingPipeline

mission_path = "/datahome/lmertens/grand_tour_dataset/2024-10-01-11-29-55"

output_dir = "./output/2024-10-01-11-29-55"

pipeline = DataLabelingPipeline(mission_path=mission_path)

pipeline.process_mission(output_dir=output_dir)

Using hyperparameters:
P_x=
[[1 0 0]
 [0 1 0]
 [0 0 1]]
P_m=
[[1 0 0]
 [0 1 0]
 [0 0 1]]
P_w=
[[1 0 0]
 [0 1 0]
 [0 0 1]]
N=
8
dt=
0.5
Loading data from /datahome/lmertens/grand_tour_dataset/2024-10-01-11-29-55 ...
Total synchronized steps: 546
Running Receding Horizon Estimator ...
Processed step 0/538
Processed step 100/538
Processed step 200/538
Processed step 300/538
Processed step 400/538
Processed step 500/538
Saving continuous labels ...
Mission processing complete.


(array([[ 1.57157854e-02, -4.26675350e-03,  8.14738532e-02],
        [ 5.36185412e-02, -6.60814773e-03,  6.40936756e-02],
        [ 1.63049556e-01,  2.17649913e-03, -3.71204352e-03],
        ...,
        [ 4.50539076e+01, -6.92599990e+01, -7.23534360e-02],
        [ 4.53292297e+01, -6.92799543e+01, -7.23534360e-02],
        [ 4.55257960e+01, -6.92942014e+01, -7.22967533e-02]],
       shape=(546, 3)),
 array([[0.95020108, 0.86302719],
        [0.93220239, 0.75454408],
        [0.92185339, 0.68035082],
        ...,
        [0.9082629 , 0.11142025],
        [0.9082629 , 0.11142025],
        [0.9082629 , 0.11142025]], shape=(546, 2)))

In [ ]:
from pathlib import Path

import numpy as np
import zarr
from tqdm.auto import tqdm

try:
    import cv2
except ImportError as exc:
    raise ImportError(
        "OpenCV is required for efficient video writing. Install with: pip install opencv-python"
    ) from exc

# Inputs
mission_path = Path("/datahome/lmertens/grand_tour_dataset/2024-10-01-11-29-55")
labels_dir = Path("./output/2024-10-01-11-29-55")
images_dir = mission_path / "images" / "zed2i_right_images"
image_topic_path = mission_path / "data" / "zed2i_right_images"

# Output video settings
video_path = labels_dir / "traversability_with_right_images.mp4"
fps = 10.0
max_frames = 200  # Set e.g. 2000 for a shorter debug video
stride = 1  # Increase (e.g. 2, 4) to speed up generation and shorten output
resize_factor = 0.3  # Set < 1.0 for faster writing and smaller files

# Load label outputs
label_ts = np.load(labels_dir / "timestamp.npy")
trav_labels = np.load(labels_dir / "trav_labels.npy")
if trav_labels.ndim == 1:
    trav_labels = trav_labels[:, None]

# Load image timestamps from zarr to align label time with the nearest camera frame
image_topic = zarr.open(str(image_topic_path), mode="r")
image_ts = np.asarray(image_topic["timestamp"])

if len(label_ts) == 0 or len(image_ts) == 0:
    raise ValueError("Empty timestamps: cannot generate video.")

# Nearest-neighbor timestamp alignment (vectorized)
insert_idx = np.searchsorted(image_ts, label_ts)
insert_idx = np.clip(insert_idx, 1, len(image_ts) - 1)
left_idx = insert_idx - 1
right_idx = insert_idx
nearest_idx = np.where(
    np.abs(label_ts - image_ts[left_idx]) <= np.abs(image_ts[right_idx] - label_ts),
    left_idx,
    right_idx,
)

# Keep only image indices for which JPEG files should exist and optionally subsample
valid = nearest_idx >= 0
frame_ids = np.where(valid)[0]
if stride > 1:
    frame_ids = frame_ids[::stride]
if max_frames is not None:
    frame_ids = frame_ids[:max_frames]

if len(frame_ids) == 0:
    raise ValueError("No valid aligned frames found.")

# Use robust normalization so bars are informative even with outliers
low = np.percentile(trav_labels, 1, axis=0)
high = np.percentile(trav_labels, 99, axis=0)
scale = np.maximum(high - low, 1e-8)


def to_unit(values: np.ndarray) -> np.ndarray:
    return np.clip((values - low) / scale, 0.0, 1.0)


# Read one frame to infer size
first_image_idx = int(nearest_idx[frame_ids[0]])
first_img_path = images_dir / f"{first_image_idx:06d}.jpeg"
first_img = cv2.imread(str(first_img_path), cv2.IMREAD_COLOR)
if first_img is None:
    raise FileNotFoundError(f"Could not read first image: {first_img_path}")

if resize_factor != 1.0:
    first_img = cv2.resize(
        first_img,
        None,
        fx=resize_factor,
        fy=resize_factor,
        interpolation=cv2.INTER_AREA,
    )

img_h, img_w = first_img.shape[:2]
panel_h = 180
frame_h = img_h + panel_h

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
# fourcc = cv2.VideoWriter_fourcc(*"h264")  # Use H.264 for better compression

writer = cv2.VideoWriter(str(video_path), fourcc, fps, (img_w, frame_h))
if not writer.isOpened():
    raise RuntimeError("Failed to open video writer.")

label_count = trav_labels.shape[1]
bar_colors = [
    (0, 200, 0),
    (0, 170, 255),
    (220, 120, 0),
    (200, 0, 160),
]

for k in tqdm(frame_ids, desc="Writing video frames"):
    image_idx = int(nearest_idx[k])
    img_path = images_dir / f"{image_idx:06d}.jpeg"
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)

    if img is None:
        continue

    if resize_factor != 1.0:
        img = cv2.resize(
            img,
            None,
            fx=resize_factor,
            fy=resize_factor,
            interpolation=cv2.INTER_AREA,
        )

    if img.shape[1] != img_w or img.shape[0] != img_h:
        img = cv2.resize(img, (img_w, img_h), interpolation=cv2.INTER_AREA)

    panel = np.full((panel_h, img_w, 3), 245, dtype=np.uint8)

    values = trav_labels[k]
    unit_values = to_unit(values)

    cv2.putText(
        panel,
        f"label_idx={k} | image_idx={image_idx} | ts={label_ts[k]}",
        (20, 28),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.62,
        (20, 20, 20),
        1,
        cv2.LINE_AA,
    )

    base_y = panel_h - 30
    max_bar_h = panel_h - 80
    spacing = img_w // max(label_count, 1)
    bar_w = max(24, min(100, spacing // 3))

    for j in range(label_count):
        center_x = int((j + 0.5) * spacing)
        x0 = center_x - bar_w // 2
        x1 = center_x + bar_w // 2

        bar_h = int(unit_values[j] * max_bar_h)
        y0 = base_y - bar_h
        color = bar_colors[j % len(bar_colors)]

        cv2.rectangle(panel, (x0, y0), (x1, base_y), color, thickness=-1)
        cv2.rectangle(panel, (x0, base_y - max_bar_h), (x1, base_y), (80, 80, 80), 1)
        cv2.putText(
            panel,
            f"m[{j}]={values[j]:.3f}",
            (max(5, x0 - 8), panel_h - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.46,
            (30, 30, 30),
            1,
            cv2.LINE_AA,
        )

    frame = np.vstack([img, panel])
    writer.write(frame)

writer.release()
print(f"Saved: {video_path}")

: 